# Hydra Configuration Framework

When training an ML model, we could use the same code using different sets of configurations and hyperparameters. Hydra is an open-source Python framework that is used in Python code to help manage configurations within the program. It has been well adopted in the data science projects.

This notebook is based on [the official Hydra sample code](https://github.com/facebookresearch/hydra/blob/main/examples/jupyter_notebooks/compose_configs_in_notebook.ipynb) for using the framework in a Jupyter notebook.

In [17]:
# Install packages
!python3 -m pip install hydra > /dev/null 2>&1

import hydra
from omegaconf import OmegaConf

## Initializing Hydra

Most hydra sample code uses the `@hydra.main` decorator to initialize hydra in your python code. It's simple and gets the job done. The problem: you need to use the decorator in a main module.

```
@hydra.main(version_base=None, config_path="conf", config_name="config")
def my_app(cfg : DictConfig) -> None:
    print(OmegaConf.to_yaml(cfg))   # Using to_yaml to pretty print

if __name__ == "__main__":
    my_app()
```

In a Jupyter notebook, instead of relying the decorator, we need to make direct calls to hydra function to initialize hydra in a book. 


### Initializing using initialize()

We initialize hydra by passing the `config_path` to the search path.

In [18]:
with hydra.initialize(version_base=None, config_path='./conf'):
    cfg = hydra.compose(overrides=['+environment=prod'])

# Using to_yaml for pretty printing cfg,
print(OmegaConf.to_yaml(cfg))

environment:
  debug: false



Note that `config_path` tells hydra where the root of the config directory. Hydra constructs a config tree by traversing all the directories recursively. It starts from `conf` directory and reads the content of subdirectories `db` and `environment`. Note hydra doesn't read the `config.yaml` file at the root,. This is why the `db` and `environment` objects are not selected, as defined in the `config.yaml`.

We need to override by passing the `+` operator, which tells hydra to instantiate an `environment` object with the value `prod`. Consequently, we load only the values matching `environment == 'prod'`. Note that the `db` values are read. For further description and example of the `+` operator, read [here](https://hydra.cc/docs/tutorials/basic/your_first_app/simple_cli/).

### Using config_name

But I want to load both the `db` and `environment` values? We need to read the `config.yaml` file by doing the following.

In [19]:
with hydra.initialize(version_base=None, config_path='./conf'):
    # The file config.yaml sets environment to dev. We can still override it to prod.
    cfg = hydra.compose(config_name='config.yaml', overrides=['environment=prod'])

# Using to_yaml for pretty printing cfg,
print(OmegaConf.to_yaml(cfg))

# Need to resolve the oc.env tag in order to read in the environment variables.
cfg = OmegaConf.to_container(cfg, resolve=True, throw_on_missing=True)
print(OmegaConf.to_yaml(cfg))

db:
  driver: snowflake
  host: xyz.snowflakecomputing.com
  authenticator: snowflake
  user: snowflake-user
  password: ${oc.env:DB_PASSWORD}
environment:
  debug: false

db:
  driver: snowflake
  host: xyz.snowflakecomputing.com
  authenticator: snowflake
  user: snowflake-user
  password: mypassword
environment:
  debug: false



## Credits and References

* [Intro to Hydra](https://hydra.cc/docs/intro/)